####Support_and_Consent

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
#Generate SupportCase records
SEED = 20261022
SUPPORT_COUNT = 500

support = spark.range(1, SUPPORT_COUNT + 1)

support = support.withColumn(
    "case_id",
    F.concat(
        F.lit("CASE-2026-"),
        F.lpad(F.col("id").cast("string"), 6, "0")
    )
)

# Assign valid customers
support = support.withColumn(
    "customer_number",
    (
        F.pmod(
            F.xxhash64("id", F.lit(SEED)),
            F.lit(1000)
        ) + 1
    ).cast("int")
)

customer_lookup = (
    spark.table("customer_master")
    .select("customer_id")
    .withColumn(
        "customer_number",
        F.row_number().over(Window.orderBy("customer_id"))
    )
)

support = support.join(
    customer_lookup,
    on="customer_number",
    how="inner"
)

# Generate issue types
support = support.withColumn(
    "issue_type",
    F.when(F.pmod(F.xxhash64("id", F.lit("issue")), F.lit(4)) == 0, "DELIVERY")
     .when(F.pmod(F.xxhash64("id", F.lit("issue")), F.lit(4)) == 1, "PAYMENT")
     .when(F.pmod(F.xxhash64("id", F.lit("issue")), F.lit(4)) == 2, "PRODUCT_QUALITY")
     .otherwise("REFUND")
)

# Generate priority
support = support.withColumn(
    "priority",
    F.when(F.pmod(F.xxhash64("id", F.lit("priority")), F.lit(3)) == 0, "HIGH")
     .when(F.pmod(F.xxhash64("id", F.lit("priority")), F.lit(3)) == 1, "MEDIUM")
     .otherwise("LOW")
)

# Generate case status
support = support.withColumn(
    "case_status",
    F.when(F.pmod(F.xxhash64("id", F.lit("status")), F.lit(3)) == 0, "OPEN")
     .when(F.pmod(F.xxhash64("id", F.lit("status")), F.lit(3)) == 1, "RESOLVED")
     .otherwise("CLOSED")
)

# Generate service cost
support = support.withColumn(
    "service_cost",
    F.round(
        F.pmod(F.xxhash64("id", F.lit("cost")), F.lit(4500)) + 500,
        2
    )
)

# Generate case dates
support = support.withColumn(
    "opened_date",
    F.date_add(
        F.lit("2025-01-01").cast("date"),
        F.pmod(F.xxhash64("id", F.lit("date")), F.lit(668)).cast("int")
    )
)

support = support.withColumn(
    "resolved_date",
    F.when(
        F.col("case_status").isin("RESOLVED", "CLOSED"),
        F.date_add(
            F.col("opened_date"),
            (F.pmod(F.xxhash64("id", F.lit("resolution")), F.lit(15)) + 1).cast("int")
        )
    )
)

support_df = support.select(
    "case_id",
    "customer_id",
    "issue_type",
    "priority",
    "case_status",
    "service_cost",
    "opened_date",
    "resolved_date"
)

display(support_df.limit(10))

In [0]:
#Validate SupportCase
print("Total Support Cases:", support_df.count())

print(
    "Duplicate Cases:",
    support_df.groupBy("case_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Invalid Customers:",
    support_df.join(
        spark.table("customer_master").select("customer_id"),
        on="customer_id",
        how="left_anti"
    ).count()
)

In [0]:
#Save SupportCase
support_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("support_case")

####Consent table

In [0]:
#Generate consent records
customers = spark.table("customer_master")

consent_df = customers.select("customer_id")

consent_df = consent_df.withColumn(
    "consent_id",
    F.concat(F.lit("CON-"), F.substring("customer_id", 5, 6))
)

consent_df = consent_df.withColumn(
    "channel",
    F.lit("ALL_MARKETING")
)

consent_df = consent_df.withColumn(
    "consent_status",
    F.when(
        F.pmod(
            F.xxhash64("customer_id", F.lit(SEED)),
            F.lit(100)
        ) < 80,
        "GRANTED"
    ).otherwise("WITHDRAWN")
)

consent_df = consent_df.withColumn(
    "effective_from",
    F.to_date(F.lit("2025-01-01"))
)

consent_df = consent_df.withColumn(
    "effective_to",
    F.lit(None).cast("date")
)

display(consent_df.limit(10))

In [0]:
#Check consent distribution
consent_df.groupBy("consent_status").count().show()

In [0]:
#Validate consent records
print("Total Consent Records:", consent_df.count())

print(
    "Duplicate Consent IDs:",
    consent_df.groupBy("consent_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
#Save Consent table
consent_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("consent")

In [0]:
%sql
SHOW TABLES;